In [2]:
import os
import pandas as pd
from pathlib import Path

In [3]:
BASE_URL = Path("../../../batch-processing/data/")

print(os.listdir(BASE_URL))

['pq', 'payment_type_lookup.csv', '.DS_Store', 'yellow_tripdata_2019-01.csv', 'head.csv', 'taxi_zone_lookup.csv', 'report', 'raw', 'green_tripdata_2019-01.csv']


In [4]:
url = BASE_URL / "pq/yellow/2021/01"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,161,262,2.40,18.50,2021-01-15 16:36:03
1,238,237,2.25,16.62,2021-01-28 06:51:29
2,132,203,5.42,18.80,2021-01-15 16:12:34
3,233,161,1.01,13.56,2021-01-21 11:49:30
4,230,229,1.00,12.95,2021-01-22 15:55:01


In [5]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int


def ride_from_row(row):
    return Ride(
        PULocationID=int(row.PULocationID),
        DOLocationID=int(row.DOLocationID),
        trip_distance=float(row.trip_distance),
        total_amount=float(row.total_amount),
        tpep_pickup_datetime=int(row.tpep_pickup_datetime.timestamp() * 1000),
    )

In [6]:
ride = ride_from_row(df.iloc[0])
ride

Ride(PULocationID=161, DOLocationID=262, trip_distance=2.4, total_amount=18.5, tpep_pickup_datetime=1610728563000)

In [7]:
import json
from kafka import KafkaProducer


def json_serializer(data):
    return json.dumps(data).encode("utf-8")


server = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer,
)

In [8]:
# test
import dataclasses

topic_name = "rides"

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

In [9]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode("utf-8")

In [10]:
# Recreating it
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer,
)

In [11]:
producer.send(topic_name, value=ride)
producer.flush()

In [12]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f"Took {(t1 - t0):.2f} seconds")

Sent: Ride(PULocationID=161, DOLocationID=262, trip_distance=2.4, total_amount=18.5, tpep_pickup_datetime=1610728563000)
Sent: Ride(PULocationID=238, DOLocationID=237, trip_distance=2.25, total_amount=16.62, tpep_pickup_datetime=1611816689000)
Sent: Ride(PULocationID=132, DOLocationID=203, trip_distance=5.42, total_amount=18.8, tpep_pickup_datetime=1610727154000)
Sent: Ride(PULocationID=233, DOLocationID=161, trip_distance=1.01, total_amount=13.56, tpep_pickup_datetime=1611229770000)
Sent: Ride(PULocationID=230, DOLocationID=229, trip_distance=1.0, total_amount=12.95, tpep_pickup_datetime=1611330901000)
Sent: Ride(PULocationID=181, DOLocationID=133, trip_distance=2.48, total_amount=22.0, tpep_pickup_datetime=1610719044000)
Sent: Ride(PULocationID=237, DOLocationID=260, trip_distance=4.2, total_amount=20.3, tpep_pickup_datetime=1611311624000)
Sent: Ride(PULocationID=162, DOLocationID=262, trip_distance=2.21, total_amount=15.96, tpep_pickup_datetime=1610441994000)
Sent: Ride(PULocationID